# 01 — EDA dữ liệu RT-IoT2022

Mục tiêu: kiểm tra chất lượng, phân tích phân bố 12 lớp, tìm dấu hiệu bất thường và đưa ra quyết định tiền xử lý có thể truy vết.

## 0. Cấu hình

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
DATA_NAME='RT_IOT2022.csv'
CANDIDATES=[]
for root in [Path.cwd(), *Path.cwd().parents]:
    CANDIDATES.extend((root/'data'/'raw'/DATA_NAME, root/DATA_NAME))
INPUT_PATH=next((p for p in CANDIDATES if p.is_file()),None)
if INPUT_PATH is None:
    raise FileNotFoundError(f'Không tìm thấy {DATA_NAME} trong data/raw. Đã kiểm tra từ: {Path.cwd()}')
PROJECT_ROOT=next((p.parent.parent.parent for p in CANDIDATES if p == INPUT_PATH), INPUT_PATH.parent.parent.parent)
OUT=PROJECT_ROOT/'reports'/'baseline'/'figures'/'baseline'/'eda';OUT.mkdir(parents=True,exist_ok=True)
TARGET='Attack_type'
df=pd.read_csv(INPUT_PATH,low_memory=False)
assert TARGET in df.columns
print('Dataset:',df.shape,'| memory MiB:',round(df.memory_usage(deep=True).sum()/1024**2,2))

## 1. Tổng quan schema, kiểu dữ liệu, mẫu quan sát

In [ ]:
display(df.head(8));display(df.dtypes.value_counts().rename('number_of_columns'))
schema=pd.DataFrame({'dtype':df.dtypes.astype(str),'nunique':df.nunique(dropna=False),'missing':df.isna().sum()})
display(schema);schema.to_csv(OUT/'schema.csv')

## 2. Missing, Inf, trùng, index thừa, cột hằng, nhãn mâu thuẫn

In [ ]:
num=df.select_dtypes(include='number'); features=[c for c in df if c not in ['Unnamed: 0',TARGET]]
quality=pd.DataFrame({'missing':df.isna().sum(),'missing_pct':df.isna().mean()*100,'unique':df.nunique(dropna=False)})
quality['inf']=0
for c in num:quality.loc[c,'inf']=int(np.isinf(df[c].to_numpy()).sum())
quality.to_csv(OUT/'data_quality.csv');display(quality.sort_values('missing',ascending=False).head(30))
conflicting=df.groupby(features,dropna=False,sort=False)[TARGET].transform('nunique').gt(1)
print('Duplicate including index:',df.duplicated().sum())
print('Duplicate excluding index:',df.drop(columns=['Unnamed: 0'],errors='ignore').duplicated().sum())
print('Feature-vector duplicates:',df.duplicated(subset=features).sum())
print('Conflicting-label rows:',int(conflicting.sum()))
print('Constant columns:',df.columns[df.nunique(dropna=False)<=1].tolist())
print('Unnamed index:',[c for c in df if c.startswith('Unnamed:')])
print('Conflicting rows by label:');display(df.loc[conflicting,TARGET].value_counts())

## 3. Phân bố 12 nhãn, mất cân bằng và lớp hiếm

In [ ]:
counts=df[TARGET].value_counts();labels=pd.DataFrame({'count':counts,'percent':counts/len(df)*100});display(labels)
labels.to_csv(OUT/'label_distribution.csv')
print('Imbalance ratio:',round(counts.max()/counts.min(),2))
fig,ax=plt.subplots(figsize=(11,6));counts.sort_values().plot.barh(ax=ax);ax.set(title='RT-IoT2022 — phân bố 12 nhãn',xlabel='Số mẫu');fig.tight_layout();fig.savefig(OUT/'label_distribution.png',dpi=140);plt.show()

## 4. Đặc trưng phân loại: proto, service, dấu gạch ngang

In [ ]:
for c in ['proto','service']:
    vc=df[c].astype('string').value_counts(dropna=False)
    display(vc.head(30).rename(c))
    vc.to_csv(OUT/f'{c}_distribution.csv')
    fig,ax=plt.subplots(figsize=(10,4));vc.head(15).sort_values().plot.barh(ax=ax);ax.set_title(f'Phân bố {c}');fig.tight_layout();fig.savefig(OUT/f'{c}_distribution.png',dpi=140);plt.show()
print('service = "-":',int(df['service'].eq('-').sum()))
print('proto × nhãn:');display(pd.crosstab(df[TARGET],df['proto'],normalize='index').round(3))

## 5. Thống kê toàn bộ đặc trưng số, zero inflation, skewness

In [ ]:
numeric=df.select_dtypes(include='number').drop(columns=['Unnamed: 0'],errors='ignore')
stats=numeric.describe(percentiles=[.01,.05,.25,.5,.75,.95,.99]).T
stats['zero_pct']=numeric.eq(0).mean()*100
stats['negative_pct']=numeric.lt(0).mean()*100
stats['skew']=numeric.skew()
stats.to_csv(OUT/'numeric_statistics.csv');display(stats.sort_values('zero_pct',ascending=False).head(25))

## 6. Quan sát Outlier theo IQR

In [ ]:
q1=numeric.quantile(.25);q3=numeric.quantile(.75);iqr=q3-q1
outliers=pd.Series({c:100*(((numeric[c]<q1[c]-1.5*iqr[c])|(numeric[c]>q3[c]+1.5*iqr[c])).mean()) for c in numeric},name='iqr_outlier_pct').sort_values(ascending=False)
display(outliers.head(30));outliers.to_csv(OUT/'outlier_rates.csv')
selected=[c for c in ['flow_duration','flow_pkts_per_sec','fwd_pkts_tot','payload_bytes_per_second'] if c in numeric]
fig,axs=plt.subplots(1,len(selected),figsize=(4*len(selected),3));axs=np.atleast_1d(axs)
for ax,c in zip(axs,selected):ax.hist(np.log1p(numeric[c].clip(lower=0)),bins=45);ax.set_title('log1p '+c)
fig.tight_layout();fig.savefig(OUT/'numeric_distributions.png',dpi=140);plt.show()

## 7. Tương quan Spearman và cặp tương quan cao (chỉ EDA)

In [ ]:
corr=numeric.loc[:,numeric.nunique()>1].corr(method='spearman');corr.to_csv(OUT/'spearman_correlation.csv')
fig,ax=plt.subplots(figsize=(12,10));im=ax.imshow(corr.to_numpy(),vmin=-1,vmax=1,cmap='coolwarm');fig.colorbar(im,ax=ax);ax.set_title('Spearman correlation (numeric)');fig.tight_layout();fig.savefig(OUT/'spearman_heatmap.png',dpi=140);plt.show()
a=corr.abs().where(np.triu(np.ones(corr.shape),k=1).astype(bool)).stack().sort_values(ascending=False)
display(a.head(20).rename('abs_spearman'));a.head(100).to_csv(OUT/'top_correlated_pairs.csv')

## 8. So sánh đặc trưng theo từng loại tấn công

In [ ]:
medians=df.groupby(TARGET)[numeric.columns].median(numeric_only=True);medians.to_csv(OUT/'class_medians.csv')
cols=[c for c in ['flow_duration','fwd_pkts_tot','bwd_pkts_tot','flow_pkts_per_sec','payload_bytes_per_second'] if c in medians]
display(medians[cols]);
fig,ax=plt.subplots(figsize=(10,5));scaled=np.log1p(medians[cols].clip(lower=0));im=ax.imshow(scaled,aspect='auto',cmap='viridis');ax.set_xticks(range(len(cols)),cols,rotation=40,ha='right');ax.set_yticks(range(len(medians)),medians.index);fig.colorbar(im,ax=ax);fig.tight_layout();fig.savefig(OUT/'class_feature_heatmap.png',dpi=140);plt.show()

### 9. PCA 2D  trực quan 

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
# Sample để giảm thời gian chạy; chỉ dùng đặc trưng số và xử lý Inf trong bản sao EDA
sample_indices=df.groupby(TARGET).apply(lambda g:g.sample(min(len(g),max(30,int(4000*len(g)/len(df)))),random_state=42).index,include_groups=False)
sample=df.loc[np.concatenate(sample_indices.to_numpy())].copy()
X=sample[numeric.columns].replace([np.inf,-np.inf],np.nan).fillna(0)
X=X.loc[:,X.nunique()>1]
X=StandardScaler().fit_transform(X)
xy=PCA(n_components=2,random_state=42).fit_transform(X)
fig,ax=plt.subplots(figsize=(10,7));
for label in sample[TARGET].unique():
 mask=(sample[TARGET]==label).to_numpy();ax.scatter(xy[mask,0],xy[mask,1],s=7,alpha=.55,label=label)
ax.legend(bbox_to_anchor=(1.02,1),loc='upper left',fontsize=8);ax.set_title('PCA 2D — EDA sample, not training transform');fig.tight_layout();fig.savefig(OUT/'pca_2d.png',dpi=140);plt.show()

## 10. EDA Findings → Preprocessing Decisions
| Bằng chứng EDA | Quyết định trong notebook 02 | Lý do |
|---|---|---|
| `Unnamed: 0` là index xuất file | Loại cột này | Không phải tín hiệu lưu lượng mạng |
| Vector đặc trưng giống nhau, nhãn mâu thuẫn | Loại toàn bộ nhóm mâu thuẫn, ghi log | Không gán nhãn tùy tiện |
| Trùng bản ghi sau bỏ index | Loại bản sao trùng, ghi log | Hạn chế đánh giá quá lạc quan |
| Cột hằng | Loại cột hằng | Không mang thông tin biến thiên |
| `service = "-"` | Đổi `unknown` | Chuẩn hóa biểu diễn, không suy diễn |
| Missing/Inf | Kiểm tra và assert | Không impute theo thống kê toàn dataset |
| 12 lớp mất cân bằng | Giữ nguyên 12 lớp | Chỉ xử lý mất cân bằng trên train nếu cần |
| Outlier trong lưu lượng mạng | Giữ nguyên | Có thể là dấu hiệu tấn công |
| Tương quan / PCA | Chỉ dùng để EDA | Không chọn đặc trưng hoặc fit scaler trước split |

**Đối chiếu đầu ra:** Notebook 02 xuất `preprocessing_audit.csv` và `RT_IOT2022_pre_split_12_classes.csv`. Các thao tác tiền xử lý đều có căn cứ ở bảng trên.

## 11. Kiểm tra leakage và đề xuất chia dữ liệu

- Không dùng `Attack_type`, nhãn suy diễn từ `Attack_type`, hoặc index làm đầu vào mô hình.
- Nếu có session/time/source metadata từ bộ thu thập, ưu tiên đánh giá split theo nhóm/thời gian để tránh leakage giữa các flow liên quan.
- Stratified 70/15/15 theo **12 nhãn gốc**, cố định `random_state`.
- Chỉ fit encoder/scaler/feature selection trên train. Validation dùng chọn mô hình, test chỉ đánh giá cuối.
- Với lớp hiếm (28–37 mẫu), báo cáo Macro-F1, Recall từng lớp và confusion matrix; tránh chỉ báo cáo accuracy.

### 12. Báo cáo EDA

In [ ]:
report=pd.DataFrame([
 ['rows',len(df)],['columns',df.shape[1]],['classes',df[TARGET].nunique()],
 ['missing_cells',int(df.isna().sum().sum())],['duplicate_rows_without_index',int(df.drop(columns=['Unnamed: 0'],errors='ignore').duplicated().sum())],
 ['conflicting_label_rows',int(conflicting.sum())],['service_dash',int(df['service'].eq('-').sum())],
 ['constant_columns',int((df.nunique(dropna=False)<=1).sum())]
],columns=['metric','value']);display(report);report.to_csv(OUT/'eda_summary.csv',index=False)
